<a href="https://colab.research.google.com/github/lijoannali/butterfly-moths-classifier/blob/main/%5BFinal%5D_HOG_%2B_SVM_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HOG and Kernel SVM Classifier
## HOG + Color Histogram + LBP + PCA Test

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.cluster import MiniBatchKMeans
from scipy.spatial.distance import cdist
import cv2
import pickle
import os
import zipfile
from skimage.feature import hog, local_binary_pattern
from collections import Counter
from sklearn.decomposition import PCA

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# CONFIGURATION
RANDOM_STATE = 4242

# Feature extraction parameters
IMAGE_SIZE = 256

# Spatial Pyramid Matching levels
SPM_LEVELS = [1, 2] # Level 0: 1x1, Level 1: 2x2

def resize_with_padding(img, target_size=(256, 256)):
    h, w = img.shape[:2]
    scale = min(target_size[0]/h, target_size[1]/w)

    new_h, new_w = int(h*scale), int(w*scale)

    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    img_resized = cv2.resize(img, (new_w, new_h), interpolation=interp)

    padded = np.zeros((target_size[0], target_size[1], 3), dtype=img.dtype)

    y_offset = (target_size[0] - new_h)//2
    x_offset = (target_size[1] - new_w)//2

    padded[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = img_resized
    return padded

def extract_hog_region(region_gray):
    return hog(
        region_gray,
        orientations=12,
        pixels_per_cell=(16,16),
        cells_per_block=(2,2),
        block_norm='L2-Hys',
        transform_sqrt=True,
        feature_vector=True
    )


def extract_lbp_hist_region(region_gray, radius=2, n_points=16):
    lbp = local_binary_pattern(region_gray, n_points, radius, method='uniform')
    n_bins = n_points + 2
    hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, n_bins + 1),
        range=(0, n_bins)
    )
    hist = hist.astype(np.float32)
    hist /= (hist.sum() + 1e-8)
    return hist


def extract_color_hist_region(region_rgb, bins=(12,12,12)):
    region_hsv = cv2.cvtColor(region_rgb, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist(
        [region_hsv],
        [0,1,2],
        None,
        list(bins),
        [0,180,0,256,0,256]
    )
    hist = hist.flatten().astype(np.float32)
    hist /= (hist.sum() + 1e-8)
    return hist


def extract_features(image_path):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        return None

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = resize_with_padding(img_rgb, (IMAGE_SIZE, IMAGE_SIZE))

    h, w = img.shape[:2]

    hog_feats = []
    color_feats = []
    lbp_feats = []

    for level in SPM_LEVELS:
        grid = level

        for i in range(grid):
            for j in range(grid):
                y1 = i * h // grid
                y2 = (i + 1) * h // grid
                x1 = j * w // grid
                x2 = (j + 1) * w // grid

                patch_rgb = img[y1:y2, x1:x2]
                patch_gray = cv2.cvtColor(patch_rgb, cv2.COLOR_RGB2GRAY)

                hog_feats.append(extract_hog_region(patch_gray))
                color_feats.append(extract_color_hist_region(patch_rgb))
                lbp_feats.append(extract_lbp_hist_region(patch_gray))

    hog_feat = np.concatenate(hog_feats).astype(np.float32)
    color_feat = np.concatenate(color_feats).astype(np.float32)
    lbp_feat = np.concatenate(lbp_feats).astype(np.float32)

    hog_feat /= (np.linalg.norm(hog_feat) + 1e-8)
    color_feat /= (np.linalg.norm(color_feat) + 1e-8)
    lbp_feat /= (np.linalg.norm(lbp_feat) + 1e-8)

    return np.concatenate([hog_feat, color_feat, lbp_feat])

# IMPROVED FEATURE NORMALIZATION
def power_normalize(features, alpha=0.5):
    """
    Power normalization: sign(x) * |x|^alpha
    """
    return np.sign(features) * np.power(np.abs(features), alpha)

def normalize_features(features, method='l2_power'):
    if method == 'l2':
        norms = np.linalg.norm(features, axis=1, keepdims=True)
        norms[norms == 0] = 1
        return features / norms

    elif method == 'l1':
        sums = np.sum(features, axis=1, keepdims=True)
        sums[sums == 0] = 1
        return features / sums

    elif method == 'l2_power':
        features = power_normalize(features, alpha=0.5)
        norms = np.linalg.norm(features, axis=1, keepdims=True)
        norms[norms == 0] = 1
        return features / norms

    return features


## Data Loading

In [4]:
# Paths
BASE_DIR = "/content/drive/Shareddrives/Elec378_Final_Project/Elec378_FP_Code"
CSV_PATH = "/content/drive/Shareddrives/Elec378_Final_Project/Elec378_FP_Code/train.csv"
TEST_IMG_DIR = "/content/drive/Shareddrives/Elec378_Final_Project/Elec378_FP_Code/test_images"
# TRAIN_IMG_DIR = "/content/drive/Shareddrives/Elec378_Final_Project/Elec378_FP_Code/train_images"
TRAIN_IMG_DIR = "/content/drive/Shareddrives/Elec378_Final_Project/Elec378_FP_Code/train_images2/train_images"  #fixed image train_000307.jpg

# Load metadata
df = pd.read_csv(CSV_PATH)
class_counts = df['TARGET'].value_counts()

# Train/validation split with stratification
X_train_files, X_val_files, y_train, y_val = train_test_split(
    df["file_name"].values,
    df["TARGET"].values,
    test_size=0.2,
    stratify=df["TARGET"].values,
    random_state=RANDOM_STATE,
)

## Feature Extraction

In [5]:
X_train_feat = []
y_train_clean = []

for fname, label in tqdm(zip(X_train_files, y_train), total=len(X_train_files)):
    path = os.path.join(TRAIN_IMG_DIR, fname)
    feat = extract_features(path)

    if feat is not None:
        X_train_feat.append(feat)
        y_train_clean.append(label)
    else:
        print("Skipped:", fname)

X_train_feat = np.array(X_train_feat, dtype=np.float32)
y_train_clean = np.array(y_train_clean)

X_val_feat = []
y_val_clean = []

for fname, label in tqdm(zip(X_val_files, y_val), total=len(X_val_files)):
    path = os.path.join(TRAIN_IMG_DIR, fname)
    feat = extract_features(path)

    if feat is not None:
        X_val_feat.append(feat)
        y_val_clean.append(label)
    else:
        print("Skipped val image:", fname)

X_val_feat = np.array(X_val_feat, dtype=np.float32)
y_val_clean = np.array(y_val_clean)
# NORMALIZE FEATURES
X_train_norm = normalize_features(X_train_feat, method='l2_power')
X_val_norm = normalize_features(X_val_feat, method='l2_power')

# Standardize (zero mean, unit variance)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_norm)
X_val_scaled = scaler.transform(X_val_norm)  # Use same scaler



pca = PCA(n_components=500, random_state=RANDOM_STATE)

X_train_final = pca.fit_transform(X_train_scaled)
X_val_final = pca.transform(X_val_scaled)

# Train final model with default params
print("\nTraining final SVM on full training set...")
final_svm = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    cache_size=1000
)
final_svm.fit(X_train_final, y_train_clean)

100%|██████████| 2519/2519 [22:53<00:00,  1.83it/s]



Training final SVM on full training set...


SVC(C=10, cache_size=1000, class_weight='balanced', random_state=4242)

## Evaluation and Prediction

In [6]:
# Training accuracy
y_train_pred = final_svm.predict(X_train_final)
train_acc = accuracy_score(y_train_clean, y_train_pred)
# Validation accuracy
y_val_pred = final_svm.predict(X_val_final)
val_acc = accuracy_score(y_val_clean, y_val_pred)
print(f"\nTraining accuracy: {train_acc:.4f}")
print(f"Validation accuracy: {val_acc:.4f}")

# Check prediction distribution
train_pred_dist = Counter(y_train_pred)
val_pred_dist = Counter(y_val_pred)
print("Classification Report (Validation):")
print(classification_report(y_val_clean, y_val_pred, zero_division=0))


# TEST SET PREDICTION
test_files = sorted(os.listdir(TEST_IMG_DIR))
print(f"Found {len(test_files)} test images")


X_test_feat = []
for fname in tqdm(test_files):
    path = os.path.join(TEST_IMG_DIR, fname)
    feat = extract_features(path)
    if feat is not None:
        X_test_feat.append(feat)
    else:
        print("Skipped test image:", fname)

X_test_feat = np.array(X_test_feat, dtype=np.float32)


X_test_norm = normalize_features(X_test_feat, method='l2_power')
X_test_scaled = scaler.transform(X_test_norm)

X_test_final = pca.transform(X_test_scaled)

y_test_pred = final_svm.predict(X_test_final)

# Create submission
image_ids = [os.path.splitext(f)[0] for f in test_files]
submission = pd.DataFrame({
    "ID": image_ids,
    "TARGET": y_test_pred
})

# Save
submission_path = "/content/drive/Shareddrives/Elec378_Final_Project/Elec378_FP_Code/submissions/submission-hog.csv"
submission.to_csv(submission_path, index=False)
print(f"\nSaved predictions to: {submission_path}")
print(submission.head(10))

# SUMMARY
print(f"""
RESULTS:
  Training accuracy: {train_acc:.4f}
  Validation accuracy: {val_acc:.4f}
""")


Training accuracy: 1.0000
Validation accuracy: 0.5625
Classification Report (Validation):
                           precision    recall  f1-score   support

                   ADONIS       0.86      0.76      0.81        25
AFRICAN GIANT SWALLOWTAIL       0.94      0.76      0.84        21
           AMERICAN SNOOT       0.48      0.57      0.52        21
                    AN 88       0.95      0.88      0.91        24
                  APPOLLO       0.52      0.46      0.49        26
     ARCIGERA FLOWER MOTH       0.41      0.54      0.46        28
                    ATALA       0.51      0.83      0.63        29
               ATLAS MOTH       0.90      0.73      0.81        26
 BANDED ORANGE HELICONIAN       0.55      0.75      0.64        28
           BANDED PEACOCK       0.83      0.79      0.81        24
        BANDED TIGER MOTH       0.80      0.59      0.68        27
            BECKERS WHITE       0.48      0.43      0.45        23
  BIRD CHERRY ERMINE MOTH       0.54 

100%|██████████| 1000/1000 [02:16<00:00,  7.31it/s]



Saved predictions to: /content/drive/Shareddrives/Elec378_Final_Project/Elec378_FP_Code/submissions/submission-hog.csv
            ID       TARGET
0  test_000001       ADONIS
1  test_000002  COPPER TAIL
2  test_000003       ADONIS
3  test_000004       ADONIS
4  test_000005       ADONIS
5  test_000006       ADONIS
6  test_000007       ADONIS
7  test_000008       ADONIS
8  test_000009  COPPER TAIL
9  test_000010       ADONIS

RESULTS:
  Training accuracy: 1.0000
  Validation accuracy: 0.5625

